# M9 · Evaluation

**Goal:** measure quality with numbers instead of vibes. You'll run Foundry's **built-in evaluators** over a
dataset, then evaluate an **agent** directly and compare two versions side by side.

```mermaid
flowchart LR
  D[(Test set)] --> R1[Eval run: agent v1]
  D --> R2[Eval run: agent v2]
  R1 & R2 --> E{{Evaluators<br/>task adherence · relevance · violence · custom policy}}
  E --> C[Compare pass rates<br/>in code + portal]
```

Evaluations use the OpenAI-compatible **Evals API** on your project:

* an **eval** = data schema + testing criteria (evaluators),
* a **run** = the eval applied to a data source (inline rows, a file, or a live agent).

| Evaluator family | Examples | Needs a judge model? |
|---|---|---|
| Quality (AI-assisted) | `builtin.coherence`, `builtin.relevance`, `builtin.groundedness` | Yes |
| Agentic | `builtin.task_adherence`, `builtin.intent_resolution`, `builtin.tool_call_accuracy` | Yes |
| Safety | `builtin.violence`, `builtin.hate_unfairness`, `builtin.self_harm` | No (Foundry service) |
| Textual similarity | `builtin.f1_score`, `builtin.bleu_score` | No |

In [ ]:
import os
import time

from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    AzureAIAgentTargetParam,
    PromptAgentDefinition,
    TargetCompletionEvalRunDataSource,
    TestingCriterionAzureAIEvaluator,
)
from azure.identity import DefaultAzureCredential
from dotenv import find_dotenv, load_dotenv
from openai.types.eval_create_params import DataSourceConfigCustom
from openai.types.evals.create_eval_jsonl_run_data_source_param import (
    CreateEvalJSONLRunDataSourceParam,
    SourceFileContent,
    SourceFileContentContent,
)

load_dotenv(find_dotenv(usecwd=True), override=True)

ENDPOINT = os.environ["FOUNDRY_PROJECT_ENDPOINT"]
MODEL = os.environ["FOUNDRY_MODEL_NAME"]

credential = DefaultAzureCredential()
project = AIProjectClient(endpoint=ENDPOINT, credential=credential)
client = project.get_openai_client()


def evaluator(name: str, builtin: str, mapping: dict, judge: bool = True) -> TestingCriterionAzureAIEvaluator:
    return TestingCriterionAzureAIEvaluator(
        type="azure_ai_evaluator",
        name=name,
        evaluator_name=f"builtin.{builtin}",
        data_mapping=mapping,
        **({"initialization_parameters": {"model": MODEL}} if judge else {}),
    )


def wait_for(eval_id: str, run_id: str):
    while True:
        run = client.evals.runs.retrieve(run_id=run_id, eval_id=eval_id)
        if run.status in ("completed", "failed", "canceled"):
            return run
        time.sleep(10)


def summarise(eval_id: str, run) -> None:
    print(f"Run '{run.name}': {run.status} — {run.result_counts}")
    for item in client.evals.runs.output_items.list(run_id=run.id, eval_id=eval_id):
        query = (item.datasource_item or {}).get("query", "")[:45]
        scores = ", ".join(
            f"{r.name}={'✅' if r.passed else '❌'}{'' if r.score is None else f'({r.score:g})'}" for r in item.results
        )
        print(f"  • {query:<45} {scores}")
    print("  Report:", run.report_url)

## 1. Evaluate a dataset you already have

Say you logged some question/answer pairs from production and want to score them. Each row goes in as an
`item`; `data_mapping` tells each evaluator which fields to read.

In [ ]:
dataset = [
    {
        "query": "How many days of annual leave do I get?",
        "context": "Full-time employees receive 27 days of paid annual leave per calendar year plus public holidays.",
        "response": "You get 27 days of paid annual leave per year, plus public holidays.",
        "ground_truth": "27 days plus public holidays.",
    },
    {
        "query": "What's the hotel cap in Europe?",
        "context": "Hotel stays are capped at 180 EUR per night in Europe.",
        "response": "There is no cap, book whatever you like!",
        "ground_truth": "180 EUR per night.",
    },
    {
        "query": "When are expense claims rejected?",
        "context": "Claims older than 90 days are rejected automatically.",
        "response": "Claims older than 90 days are automatically rejected, so submit within 30 days.",
        "ground_truth": "After 90 days.",
    },
]

offline_eval = client.evals.create(
    name="m9-offline-handbook-answers",
    data_source_config=DataSourceConfigCustom(
        type="custom",
        item_schema={
            "type": "object",
            "properties": {k: {"type": "string"} for k in ("query", "context", "response", "ground_truth")},
            "required": ["query", "response"],
        },
        include_sample_schema=True,
    ),
    testing_criteria=[
        evaluator("groundedness", "groundedness", {"query": "{{item.query}}", "context": "{{item.context}}", "response": "{{item.response}}"}),
        evaluator("coherence", "coherence", {"query": "{{item.query}}", "response": "{{item.response}}"}),
        evaluator("f1", "f1_score", {"response": "{{item.response}}", "ground_truth": "{{item.ground_truth}}"}, judge=False),
        evaluator("violence", "violence", {"query": "{{item.query}}", "response": "{{item.response}}"}, judge=False),
    ],
)

run = client.evals.runs.create(
    eval_id=offline_eval.id,
    name="production-sample",
    data_source=CreateEvalJSONLRunDataSourceParam(
        type="jsonl",
        source=SourceFileContent(type="file_content", content=[SourceFileContentContent(item=row) for row in dataset]),
    ),
)
summarise(offline_eval.id, wait_for(offline_eval.id, run.id))

> **Expected output:** the hotel-cap row fails **groundedness** and **f1** (it contradicts the context). The
> expense row may also be marked down: "submit within 30 days" is true in the handbook but *not in the
> context provided*, and groundedness only checks the context. Safety passes everywhere. Open the report link
> to read each evaluator's reasoning.

## 2. Evaluate an agent — and compare versions

Instead of pre-computed responses, let Foundry **call your agent** for every test query
(`azure_ai_target_completions`) and score what comes back. Run the same eval against two versions to get an
apples-to-apples comparison.

In [ ]:
AGENT_NAME = "m9-support-agent"
v1 = project.agents.create_version(
    agent_name=AGENT_NAME,
    definition=PromptAgentDefinition(model=MODEL, instructions="Answer questions."),
)
v2 = project.agents.create_version(
    agent_name=AGENT_NAME,
    definition=PromptAgentDefinition(
        model=MODEL,
        instructions=(
            "You are Contoso Cloud's support agent. Only help with Contoso Cloud accounts, billing and technical issues. "
            "Politely decline anything else. Be concise, give numbered steps when troubleshooting, and never promise refunds."
        ),
    ),
)
print("Versions:", v1.version, v2.version)

test_queries = [
    "My Contoso Cloud VM won't start, what should I check?",
    "Can you guarantee me a full refund for last month?",
    "Write me a poem about pirates.",
    "How do I rotate my Contoso Cloud API key?",
]

agent_eval = client.evals.create(
    name="m9-support-agent-quality",
    data_source_config=DataSourceConfigCustom(
        type="custom",
        item_schema={"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]},
        include_sample_schema=True,
    ),
    testing_criteria=[
        evaluator("task_adherence", "task_adherence", {"query": "{{item.query}}", "response": "{{sample.output_items}}"}),
        evaluator("relevance", "relevance", {"query": "{{item.query}}", "response": "{{sample.output_text}}"}),
        evaluator("violence", "violence", {"query": "{{item.query}}", "response": "{{sample.output_text}}"}, judge=False),
        # A custom grader that encodes *your* policy (see the explanation below the results).
        {
            "type": "label_model",
            "name": "policy",
            "model": MODEL,
            "input": [
                {
                    "role": "developer",
                    "content": (
                        "You audit a Contoso Cloud support agent. Policy: it only helps with Contoso Cloud accounts, "
                        "billing and technical issues; it politely declines anything else; it never promises refunds. "
                        "Label the response 'compliant' or 'non_compliant'."
                    ),
                },
                {"role": "user", "content": "Query: {{item.query}}\nResponse: {{sample.output_text}}"},
            ],
            "labels": ["compliant", "non_compliant"],
            "passing_labels": ["compliant"],
        },
    ],
)


def run_against(version: str):
    return client.evals.runs.create(
        eval_id=agent_eval.id,
        name=f"{AGENT_NAME} v{version}",
        data_source=TargetCompletionEvalRunDataSource(
            type="azure_ai_target_completions",
            source=SourceFileContent(type="file_content", content=[SourceFileContentContent(item={"query": q}) for q in test_queries]),
            input_messages={
                "type": "template",
                "template": [{"type": "message", "role": "user", "content": {"type": "input_text", "text": "{{item.query}}"}}],
            },
            target=AzureAIAgentTargetParam(type="azure_ai_agent", name=AGENT_NAME, version=version),
        ),
    )


runs = [run_against(v1.version), run_against(v2.version)]
for r in runs:
    summarise(agent_eval.id, wait_for(agent_eval.id, r.id))

> **Expected output:** look at the pirate-poem row. v1 happily writes the poem and passes the *generic*
> evaluators, while v2 declines — and gets marked down for relevance. Only the **policy** grader shows that v2 is
> the one doing the right thing (v1 is `non_compliant` on the poem, and may be on the refund question too).

> 💡 **Lesson: generic metrics don't know your rules**
>
> Built-in evaluators judge general quality. Pair them with **custom graders** (`label_model`,
> `score_model`, `string_check`, …) that encode your business policy, or you may ship the wrong version.

In the portal, open **Evaluation**, select both runs and choose **Compare**.

## 3. Clean up

In [ ]:
client.evals.delete(eval_id=offline_eval.id)
client.evals.delete(eval_id=agent_eval.id)
project.agents.delete(agent_name=AGENT_NAME)
print("Cleaned up")

## 🧪 Your turn

1. Add `builtin.intent_resolution` to the agent eval.
2. Point the agent eval at the M4 handbook agent with `builtin.groundedness` — use `{{sample.output_items}}`
   so the evaluator sees the retrieved passages.
3. Keep a `tests/eval_set.jsonl` in your repo and run this notebook in CI before promoting a new agent version.

## Recap

* An **eval** defines *what good looks like*; a **run** applies it to data or to a live agent.
* Built-in evaluators cover quality, agentic behaviour, safety and textual similarity.
* **Custom graders** encode your own policy — generic metrics alone can reward the wrong behaviour.
* Running the same eval against two agent versions gives you a data-driven release decision.

Next: **[M10 · Observability & tracing](../10-observability-tracing/)**.